# CASE 5: Single-Node Boosting Pipeline Architecture


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("Imports processed successfully.")



## 1. Subsampled Global Pool Ingestion
Loading downsampled dataset natively on-node, and executing a unified 70/30 train/test split. Every boosting algorithm will measure against this identical cross-section.


In [ ]:
df = pd.read_csv("Global_Pool_Subsampled_Train.csv")
X = df.drop(columns=['Label'])
y = df['Label']

# Secure Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

print(f"Data Loaded. Global Train Base Matrix Dimensions: {X_train.shape}")
print(f"Global Test Validation Base Matrix Dimensions: {X_test.shape}")



## 2. Baseline Naive LightGBM (Unmodified)
Used strictly for the target Null-Hypothesis validation check. (Default parameters, no class weights).


In [ ]:
from lightgbm import LGBMClassifier

print("Evaluating unmodified Naive LightGBM Baseline...")
gb_base = LGBMClassifier(random_state=42, n_estimators=50, n_jobs=-1, verbose=-1)
gb_base.fit(X_train, y_train)

y_pred_base = gb_base.predict(X_test)

# Extract Baseline MCC for statistical testing reference later
mcc_base_val = matthews_corrcoef(y_test, y_pred_base)
print(f"Baseline Naive LightGBM MCC Yield: {mcc_base_val:.4f}")



---


## 3. Algorithm 1: XGBoost (Intervening mapped `scale_pos_weight`/Class Weights)


In [ ]:
from xgboost import XGBClassifier

print(f"1. BEFORE XGBOOST INTERVENTION - Training Shape: {X_train.shape} | Training Targets: {y_train.shape}")

# Mathematically derive scale_pos_weight logic for MULTICLASS arrays dynamically
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
weight_dict = dict(zip(classes, class_weights))

# Convert class weights to discrete sample weights for precise XGB injection
sample_weights_xgb = np.array([weight_dict[lbl] for lbl in y_train])

model_xgb = XGBClassifier(random_state=42, n_estimators=100, n_jobs=-1, eval_metric='logloss')

# Fit explicitly supplying our simulated scale_pos_weights
model_xgb.fit(X_train, y_train, sample_weight=sample_weights_xgb)

print(f"2. AFTER XGBOOST INTERVENTION - The physical matrix retains its shape: {X_train.shape}, however, sample_weights of dimension {sample_weights_xgb.shape} mapped mathematical densities over it!")

# Prediction & CSV Generation
y_pred_xgb = model_xgb.predict(X_test)
res_xgb = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_xgb})
res_xgb.to_csv("results_xgboost.csv", index=False)
print("-> Exported predictions to 'results_xgboost.csv'")



---


## 4. Algorithm 2: LightGBM (Native `is_unbalance=True` mapping via `class_weight`)


In [ ]:
from lightgbm import LGBMClassifier

print(f"1. BEFORE LIGHTGBM INTERVENTION - Training Shape: {X_train.shape} | Training Targets: {y_train.shape}")

# is_unbalance mechanics naturally map to 'balanced' class weights in multiclass setups.
model_lgb = LGBMClassifier(
    random_state=42, 
    n_estimators=100, 
    class_weight='balanced', 
    min_child_samples=5, # Lowered specifically to catch minor infiltration subsets
    n_jobs=-1,
    verbose=-1
)
model_lgb.fit(X_train, y_train)

print(f"2. AFTER LIGHTGBM INTERVENTION - The engine altered gradient derivation rules inherently rather than restructuring the {X_train.shape} frame.")

y_pred_lgb = model_lgb.predict(X_test)
res_lgb = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_lgb})
res_lgb.to_csv("results_lightgbm.csv", index=False)
print("-> Exported predictions to 'results_lightgbm.csv'")



---


## 5. Algorithm 3: CatBoost (`auto_class_weights='Balanced'`)


In [ ]:
from catboost import CatBoostClassifier

print(f"1. BEFORE CATBOOST INTERVENTION - Training Shape: {X_train.shape} | Training Targets: {y_train.shape}")

model_cat = CatBoostClassifier(
    random_state=42,
    iterations=100,
    auto_class_weights='Balanced',
    verbose=False,
    thread_count=-1
)
model_cat.fit(X_train, y_train)

print(f"2. AFTER CATBOOST INTERVENTION - Trees were grown treating minority categories effectively as if they occupied equally sized slices of {X_train.shape}.")

y_pred_cat = model_cat.predict(X_test).ravel()
res_cat = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_cat})
res_cat.to_csv("results_catboost.csv", index=False)
print("-> Exported predictions to 'results_catboost.csv'")



---


## 6. Algorithm 4: AdaBoost via SMOTE Generative Synthesis


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE

print(f"1. BEFORE SMOTE + ADABOOST - Physical Training Shape: {X_train.shape} | Training Targets: {y_train.shape}")

# Here we actually change the literal shape of the training subset by extrapolating synthetics
# Using K=4 neighbors ensures it doesn't crash on extremely tight ultra-rarities.
smote = SMOTE(k_neighbors=4, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"2. AFTER SMOTE + ADABOOST - The physical data matrix HAS GROWN! New Trained Shape: {X_resampled.shape} | Targeting Dimensions: {y_resampled.shape}")

model_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=50,
    random_state=42
)
model_ada.fit(X_resampled, y_resampled)

y_pred_ada = model_ada.predict(X_test)
res_ada = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_ada})
res_ada.to_csv("results_adaboost.csv", index=False)
print("-> Exported predictions to 'results_adaboost.csv'")



---


## 7. Algorithm 5: HistGradientBoosting (Explicit Target Weight Matrices)


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

print(f"1. BEFORE HISTGRADIENT INTERVENTION - Training Shape: {X_train.shape} | Training Targets: {y_train.shape}")

model_hist = HistGradientBoostingClassifier(random_state=42, max_iter=100)

sample_weights_hist = compute_sample_weight(class_weight='balanced', y=y_train)
model_hist.fit(X_train, y_train, sample_weight=sample_weights_hist)

print(f"2. AFTER HISTGRADIENT INTERVENTION - Physical shape remained {X_train.shape}. Binning heuristics prioritized gradient pathways linked to minority indexes via strict vector manipulation.")

y_pred_hist = model_hist.predict(X_test)
res_hist = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_hist})
res_hist.to_csv("results_histgradient.csv", index=False)
print("-> Exported predictions to 'results_histgradient.csv'")



---


## 8. Master Performance Evaluation & Statistical Verification
Reading all 5 distinct generated files to compile accuracy metrics natively.


In [ ]:
# Identify target extraction
classes_global = np.unique(y_test)
infiltration_lbl = [lbl for lbl in classes_global if 'infiltr' in str(lbl).lower()]
heartbleed_lbl = [lbl for lbl in classes_global if 'heartbleed' in str(lbl).lower()]

files = {
    'XGBoost': 'results_xgboost.csv',
    'LightGBM': 'results_lightgbm.csv',
    'CatBoost': 'results_catboost.csv',
    'AdaBoost_SMOTE': 'results_adaboost.csv',
    'HistGradient': 'results_histgradient.csv'
}

evaluations = []

for model_name, csv_file in files.items():
    res = pd.read_csv(csv_file)
    yt = res['y_true']
    yp = res['y_pred']
    
    acc = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, average='weighted', zero_division=0)
    rec = recall_score(yt, yp, average='weighted', zero_division=0)
    f1 = f1_score(yt, yp, average='weighted', zero_division=0)
    macro_f1 = f1_score(yt, yp, average='macro', zero_division=0)
    mcc = matthews_corrcoef(yt, yp)
    
    # Check infiltration specifics
    class_recalls = recall_score(yt, yp, average=None, labels=classes_global, zero_division=0)
    c_rec_dict = dict(zip(classes_global, class_recalls))
    
    inf_rec = c_rec_dict[infiltration_lbl[0]] if infiltration_lbl else 0
    hrt_rec = c_rec_dict[heartbleed_lbl[0]] if heartbleed_lbl else 0
    
    evaluations.append({
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'Weighted F1': f1,
        'Macro F1': macro_f1,
        'MCC': mcc,
        'Infiltration Recall': inf_rec,
        'Heartbleed Recall': hrt_rec
    })

eval_df = pd.DataFrame(evaluations)
print("\n================ COMPARATIVE METRICS ================\n")
display(eval_df)



In [ ]:
# Visualize MCC vs Macro F1
subset = eval_df[['Model', 'MCC', 'Macro F1']].set_index('Model')
subset.plot(kind='bar', figsize=(10, 5), colormap='coolwarm')
plt.title("Boosting Optimization Stability: MCC vs Macro F1")
plt.ylabel("Evaluation Score")
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Define Best Model
best_mcc_row = eval_df.loc[eval_df['MCC'].idxmax()]
print(f"\n🏆 WINNING TOP PERFORMER -> {best_mcc_row['Model']}")
print(f"   Max MCC: {best_mcc_row['MCC']:.4f}")
print(f"   Infiltration Capture Rate: {best_mcc_row['Infiltration Recall'] * 100:.2f}%")

